# MCP Server

Starts the FastMCP server that provides agent card discovery (via embeddings), travel data queries, and Google Places lookup.

In [1]:
import json
import logging
import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import requests
import weave
from openai import OpenAI
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

AGENT_CARDS_DIR = 'agent_cards'
EMBEDDING_MODEL = 'text-embedding-3-small'
SQLITE_DB = 'travel_agency.db'
PLACES_API_URL = 'https://places.googleapis.com/v1/places:searchText'
HOST, PORT = 'localhost', 10100

oai = OpenAI()

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


2026-04-15 14:13:49,968 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-15 14:13:50,228 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-15 14:13:50,587 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-15 14:13:50,661 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-15 14:13:50,741 INFO httpx: HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
2026-04-15 14:13:50,870 INFO httpx: HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"


weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


2026-04-15 14:13:50,886 INFO weave.trace.init_message: Logged in as Weights & Biases user: paulbruffett.
View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave
Weave initialized: pbruffett/a2a-lab


In [2]:
def load_agent_cards():
    card_uris, agent_cards = [], []
    for filename in sorted(os.listdir(AGENT_CARDS_DIR)):
        if filename.endswith('.json'):
            with open(Path(AGENT_CARDS_DIR) / filename) as f:
                data = json.load(f)
            card_uris.append(f'resource://agent_cards/{Path(filename).stem}')
            agent_cards.append(data)
    return card_uris, agent_cards


def build_agent_card_embeddings():
    card_uris, agent_cards = load_agent_cards()
    texts = [json.dumps(card) for card in agent_cards]
    result = oai.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in result.data]
    print(f'Loaded {len(card_uris)} agent cards with embeddings')
    return card_uris, agent_cards, np.array(embeddings)


card_uris, agent_cards, card_embeddings = build_agent_card_embeddings()

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d92fe-46c4-72e8-82a1-d34d8c83bc73


2026-04-15 14:13:51,050 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d92fe-46c4-72e8-82a1-d34d8c83bc73
2026-04-15 14:13:51,880 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Loaded 5 agent cards with embeddings


In [3]:
mcp = FastMCP('agent-cards', host=HOST, port=PORT)


@mcp.tool(name='find_agent', description='Find the most relevant agent card for a query.')
@weave.op()
def find_agent(query: str) -> str:
    logger.info('[mcp.find_agent] query=%r', query)
    result = oai.embeddings.create(input=[query], model=EMBEDDING_MODEL)
    query_embedding = result.data[0].embedding
    scores = card_embeddings @ query_embedding
    chosen = agent_cards[int(np.argmax(scores))]
    logger.info('[mcp.find_agent] chosen=%s', chosen.get('name'))
    return json.dumps(chosen)


@mcp.tool()
@weave.op()
def query_places_data(query: str):
    """Query Google Places."""
    logger.info('[mcp.query_places_data] query=%r', query)
    api_key = os.getenv('GOOGLE_PLACES_API_KEY')
    if not api_key:
        return {'places': []}
    headers = {
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'places.id,places.displayName,places.formattedAddress',
        'Content-Type': 'application/json',
    }
    try:
        resp = requests.post(PLACES_API_URL, headers=headers, json={'textQuery': query, 'languageCode': 'en', 'maxResultCount': 10})
        resp.raise_for_status()
        data = resp.json()
        logger.info('[mcp.query_places_data] places=%d', len(data.get('places', [])))
        return data
    except Exception as e:
        logger.warning('[mcp.query_places_data] error=%s', e)
        return {'places': []}


@mcp.tool()
@weave.op()
def query_travel_data(query: str) -> dict:
    """Query the travel SQLite database (flights, hotels, rental_cars). Only SELECT queries."""
    logger.info('[mcp.query_travel_data] query=%r', query)
    if not query or not query.strip().upper().startswith('SELECT'):
        raise ValueError(f'Invalid query: {query}')
    try:
        with sqlite3.connect(SQLITE_DB) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.cursor().execute(query).fetchall()
            result = json.dumps({'results': [dict(row) for row in rows]})
            logger.info('[mcp.query_travel_data] rows=%d', len(rows))
            return result
    except Exception as e:
        logger.warning('[mcp.query_travel_data] error=%s', e)
        return {'error': str(e)}


@mcp.resource('resource://agent_cards/list', mime_type='application/json')
def get_agent_cards() -> dict:
    return {'agent_cards': list(card_uris)}


@mcp.resource('resource://agent_cards/{card_name}', mime_type='application/json')
def get_agent_card(card_name: str) -> dict:
    uri = f'resource://agent_cards/{card_name}'
    matches = [card for u, card in zip(card_uris, agent_cards) if u == uri]
    return {'agent_card': matches}

In [ ]:
print(f'MCP Server running at http://{HOST}:{PORT}')
await mcp.run_sse_async()

MCP Server running at http://localhost:10100


INFO:     Started server process [4760]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10100 (Press CTRL+C to quit)


2026-04-15 14:13:52,782 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
2026-04-15 14:13:54,362 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:53460 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:53461 - "POST /messages/?session_id=d5943745252344fbad0218c4a878207d HTTP/1.1" 202 Accepted
INFO:     ::1:53461 - "POST /messages/?session_id=d5943745252344fbad0218c4a878207d HTTP/1.1" 202 Accepted
INFO:     ::1:53461 - "POST /messages/?session_id=d5943745252344fbad0218c4a878207d HTTP/1.1" 202 Accepted
2026-04-15 14:14:00,611 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d92fe-6c24-719c-b455-14d623dc4662


2026-04-15 14:14:00,616 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d92fe-6c24-719c-b455-14d623dc4662
2026-04-15 14:14:01,916 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:53470 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:53471 - "POST /messages/?session_id=7513ad5076df4f948e89ca70c7826a6e HTTP/1.1" 202 Accepted
INFO:     ::1:53471 - "POST /messages/?session_id=7513ad5076df4f948e89ca70c7826a6e HTTP/1.1" 202 Accepted
INFO:     ::1:53471 - "POST /messages/?session_id=7513ad5076df4f948e89ca70c7826a6e HTTP/1.1" 202 Accepted
2026-04-15 14:14:11,593 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


...

2026-04-15 21:48:15,311 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949e-4bce-7776-a7a0-97cbcd1093ca
2026-04-15 21:48:16,804 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:55385 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:55386 - "POST /messages/?session_id=cbce583d7c2b4ead8a1537a240b3c31d HTTP/1.1" 202 Accepted
INFO:     ::1:55386 - "POST /messages/?session_id=cbce583d7c2b4ead8a1537a240b3c31d HTTP/1.1" 202 Accepted
INFO:     ::1:55386 - "POST /messages/?session_id=cbce583d7c2b4ead8a1537a240b3c31d HTTP/1.1" 202 Accepted
2026-04-15 21:48:28,747 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949e-804b-78a5-bb59-c336d33ebab1


2026-04-15 21:48:28,749 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949e-804b-78a5-bb59-c336d33ebab1
2026-04-15 21:48:29,665 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:55394 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:55395 - "POST /messages/?session_id=e5cb8ed199de420694da411a14e6c7fe HTTP/1.1" 202 Accepted
INFO:     ::1:55395 - "POST /messages/?session_id=e5cb8ed199de420694da411a14e6c7fe HTTP/1.1" 202 Accepted
INFO:     ::1:55395 - "POST /messages/?session_id=e5cb8ed199de420694da411a14e6c7fe HTTP/1.1" 202 Accepted
2026-04-15 21:49:10,227 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949f-2253-70ed-ba2f-4dd3e52060f8


2026-04-15 21:49:10,229 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d949f-2253-70ed-ba2f-4dd3e52060f8
2026-04-15 21:49:11,367 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
